# Benchmark results

Reads `benchmarks/results.csv` from `benchmarks/run_benchmark.py` (3 jobs x 4 modes x 3 runs). Analysis sits after each cell, not before.

## Setup

In [ ]:
%matplotlib inline
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

RESULTS = Path('../benchmarks/results.csv')
MODE_ORDER = ['local', 'cluster-1', 'cluster-2', 'cluster-3']
print(RESULTS.resolve())

## 1. Raw runs

In [ ]:
df = pd.read_csv(RESULTS)
print(f'{len(df)} rows')
df.head()

One row = one spark-submit from start to exit. The `seconds` column includes container start, JVM boot, executor handshake (cluster modes), the actual work, and teardown. That is the honest user-facing wall clock for 'does distribution help'.

In [ ]:
df.groupby(['mode', 'job'])['run'].count().unstack('job')

3 runs per cell, no gaps. Safe to take medians.

## 2. Median per cell

In [ ]:
medians = (df
           .groupby(['mode', 'job'])['seconds']
           .median()
           .unstack('job')
           .reindex(MODE_ORDER)
           .round(1))
medians

- `local` wins every job. On one host the cluster has no extra CPUs, disks, or RAM to exploit, only the handshake and cross-JVM shuffle tax to pay.
- `q_speed_by_hour.py` is ~22 s flat across all modes. The actual work is sub-second; 22 s is pure per-submission overhead.
- `clean.py` cluster-1 is 2x slower than local (191.6 vs 93.7). Full distribution tax, zero parallelism gain.
- `clean.py` scales monotonically: 191.6 -> 120.0 -> 103.5 (37% then 14% faster). Classic sub-linear speedup.
- Even cluster-3 doesn't beat local for `clean.py` (103.5 vs 93.7). Distribution overhead never pays off on one machine at this data size.
- `q_trip_distance.py` shows the same shape smaller: 19.6 (local) -> 32.2 (c1) -> 25.2 (c2) -> 25.4 (c3).

## 3. Chart: median per job

In [ ]:
ax = medians.T.plot(kind='bar', figsize=(10, 5))
ax.set_ylabel('Median wall-clock time (seconds)')
ax.set_title('VED pipeline: median runtime by job and mode')
ax.grid(axis='y', linestyle='--', alpha=0.3)
ax.legend(title='mode')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

- `clean.py`: cluster-1 bar towers over the rest. Three cluster bars descend left to right as workers grow.
- `q_speed_by_hour.py`: four bars nearly equal. Workload too small for cluster size to matter.
- `q_trip_distance.py`: a milder version of the `clean.py` pattern. Local wins; cluster-2 and cluster-3 converge.

## 4. Chart: scaling view

In [ ]:
ax = medians.plot(kind='bar', figsize=(10, 5))
ax.set_ylabel('Median wall-clock time (seconds)')
ax.set_title('VED pipeline: scaling across modes')
ax.grid(axis='y', linestyle='--', alpha=0.3)
ax.legend(title='job')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

**Scaling left to right (adding workers)**

- `clean.py`: 93.7 (local) -> 191.6 -> 120.0 -> 103.5. The jump from local to cluster-1 is the handshake tax. Each extra worker eats into that tax by splitting the work.
- `q_speed_by_hour.py`: flat at ~22 s. Nothing to parallelise.
- `q_trip_distance.py`: 19.6 -> 32.2 -> 25.2 -> 25.4. Workers help up to 2; cluster-3 adds nothing.

**With 4+ workers on the same machine**

- cluster-4: likely ~95-100 s for `clean.py`, 5-10% off cluster-3.
- cluster-5: ~92-96 s, roughly matching local.
- Beyond 5: added workers start costing more than they save (host-core contention, GC pressure, shuffle grows quadratically).
- Asymptote is set by what doesn't distribute: driver coordination, single-datanode disk, shuffle network.

**On a real multi-machine cluster**

- Each worker on its own host = own disk, own CPU, no contention. Near-linear speedup until the network saturates.
- Multiple datanodes = replicated blocks, parallel reads, I/O bottleneck gone.
- `clean.py` cluster-3 would be roughly `local_time / 2.5-3`, not `local_time * 1.1` as here.

**Headline**

On a single-host Docker cluster, distribution's overhead almost always exceeds its parallelism gain. The cluster functions correctly and scales in the expected direction (`clean.py` 46% faster from 1 to 3 workers), but it does not beat `local[*]` on this hardware. A cost-of-distribution measurement, not a performance win.